In [ ]:
# Cell 1 — Install dependencies
%pip install bertopic FlagEmbedding

In [0]:
# Cell 2 — Imports + setup: load evidence, embeddings, and ground truth
# ============================================================================
# ROOT-CAUSE ANALYSIS  (STEP 1 of 2) — setup & load evidence
#
# For each deviation we ask an LLM to read the highest-level generated context
# (contextual_retrieval_text_full = deterministic refs + llm context + source)
# alongside the raw deviation free text (source_free_text_full) and judge the
# most likely ROOT CAUSE.
#
# Served via the Databricks Foundation Model API (OpenAI-compatible) using the
# notebook's own workspace token — no external API key, same as the glossary.
# ============================================================================
import json
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import plotly.express as px
from openai import OpenAI
from pyspark.sql import functions as F, types as T

from FlagEmbedding import BGEM3FlagModel
from bertopic import BERTopic
from bertopic.backend import BaseEmbedder
from hdbscan import HDBSCAN
from umap import UMAP
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.preprocessing import LabelEncoder

CATALOG = "us_gmsgq_dev"
ALYT    = "gms_us_alyt"
MART    = "gms_us_mart"

EMB_TABLE    = f"{CATALOG}.{ALYT}.deviation_embeddings"             # text + BGE-M3 vectors
SOURCE_TABLE = f"{CATALOG}.{MART}.tw_deviation_data_formatted_rdq"  # ground-truth root cause
RC_TABLE     = f"{CATALOG}.{ALYT}.deviation_root_cause"             # output

MODEL_NAME  = "databricks-gpt-5-4-mini"  # Databricks GPT-5-4 Mini endpoint
PARALLEL    = 20                        # provisioned endpoint handles higher concurrency
LIMIT_N     = None                      # set to e.g. 20 to smoke-test on a few events first

# Databricks Foundation Model APIs are OpenAI-compatible. Authenticate with the
# notebook's own workspace token and point at the serving-endpoints base URL.
DBX_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
DBX_HOST  = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
client = OpenAI(api_key=DBX_TOKEN, base_url=f"{DBX_HOST}/serving-endpoints")

def _strip_for_tfidf(text):
    """Strip 'Column_Name: ' scaffolding so c-TF-IDF sees only content."""
    text = re.sub(
        r"\b(Event_Title|Event_Description|Impact_Assessment|Quality_Final_Assessment"
        r"|Root_Cause_Category|Root_Cause_SubCategory|Action_Text)\s*:",
        "", text
    )
    return text.strip()

# ---- Load evidence once: text + precomputed BGE-M3 vectors (fine tier) ------
emb_pdf = (
    spark.table(EMB_TABLE)
    .select("pr_id", "source_free_text_full", "contextual_retrieval_text_full", "embedding")
    .toPandas()
)
emb_pdf["pr_id"] = emb_pdf["pr_id"].astype(str)
for c in ["source_free_text_full", "contextual_retrieval_text_full"]:
    emb_pdf[c] = emb_pdf[c].fillna("")

# LLM classifier evidence (optionally smoke-tested on the first LIMIT_N events)
rc_pdf = emb_pdf[["pr_id", "source_free_text_full", "contextual_retrieval_text_full"]]
if LIMIT_N:
    rc_pdf = rc_pdf.head(LIMIT_N).copy()
print(f"Loaded {len(rc_pdf):,} deviations for root-cause analysis")

# c-TF-IDF docs (clean text) + clustering vectors (all events)
rc_docs = [_strip_for_tfidf(t) for t in emb_pdf["source_free_text_full"].tolist()]
rc_embs = np.array(emb_pdf["embedding"].tolist(), dtype=np.float32)
print(f"Embeddings loaded: {rc_embs.shape}  |  Docs: {len(rc_docs):,}")

# ---- Load ground-truth Root_Cause_Category from source mart (date/row grain) -
gt_pdf = (
    spark.table(SOURCE_TABLE)
    .select(
        F.col("Event_Number").cast("string").alias("pr_id"),
        "Root_Cause_Category",
        "Root_Cause_SubCategory",
    )
    .toPandas()
)
gt_pdf["Root_Cause_Category"] = gt_pdf["Root_Cause_Category"].fillna("")
gt_pdf["Root_Cause_SubCategory"] = gt_pdf["Root_Cause_SubCategory"].fillna("")

gt_categories = gt_pdf[gt_pdf["Root_Cause_Category"].str.strip() != ""]["Root_Cause_Category"].unique().tolist()
print(f"\nGround-truth root-cause categories ({len(gt_categories)}): {sorted(gt_categories)}")

# Single ground-truth lookup (pr_id -> Root_Cause_Category), reused across cells
gt_lookup = gt_pdf.drop_duplicates(subset="pr_id").set_index("pr_id")["Root_Cause_Category"]

In [0]:
# Cell 3 — RCA taxonomy, prompts, and the LLM classifier
# ============================================================================
# ROOT-CAUSE ANALYSIS  (STEP 2 of 2) — taxonomy, prompts, classifier
#
# A fixed category taxonomy keeps judgements consistent across events. The LLM
# returns strict JSON, which the classifier parses into a RootCauseJudgement.
# ============================================================================

# Ishikawa Fishbone (6M) root-cause taxonomy — aligned with ground-truth categories
# in the source table (Root_Cause_Category column). Each main category maps to one
# of the six fishbone "bones"; sub-categories provide finer granularity.
ROOT_CAUSE_CATEGORIES = [
    "Man (Human Factors)",        # Human error, fatigue, distraction, violation
    "Method",                     # SOP gap, process design, procedure inadequacy
    "Machine",                    # Equipment failure, IT system, instrument malfunction
    "Material",                   # Raw material, component, sample, supplier issue
    "Measurement",                # Calibration, testing error, monitoring, documentation
    "Environment",                # Environmental conditions, contamination, scheduling/timing
    "Root Cause - Not Identified",  # Insufficient evidence to determine
]

# Sub-categories the LLM can optionally pick as contributing factors
ISHIKAWA_SUBCATEGORIES = {
    "Man (Human Factors)": ["Distraction", "Error", "Violation/Deliberate action",
                           "Training gap", "Competency", "Communication/Handoff"],
    "Method": ["SOP gap or missing step", "Process design flaw",
              "Inadequate validation/qualification", "Procedure not followed"],
    "Machine": ["Equipment failure", "IT system/software", "Instrument calibration",
               "Computerized system validation"],
    "Material": ["Raw material defect", "Component failure", "Sample integrity",
                "Vendor/Supplier/CRO issue"],
    "Measurement": ["Testing error", "Documentation error", "Monitoring gap",
                    "Data integrity issue"],
    "Environment": ["Environmental excursion", "Contamination", "Facility/utility",
                    "Scheduling/timing"],
}

SYSTEM_RCA_PROMPT = """You are a clinical-trial quality investigator performing root-cause analysis (RCA)
on pharmaceutical deviation records at Takeda.

Use the ISHIKAWA FISHBONE (6M) framework to reason about the root cause:
  - Man (Human Factors): Was the deviation caused by a person? (error, distraction,
    fatigue, violation, training gap, miscommunication)
  - Method: Was it a process or procedure issue? (SOP gap, inadequate validation,
    process design flaw, procedure not followed)
  - Machine: Was it equipment or technology? (instrument failure, IT system,
    software bug, calibration issue)
  - Material: Was it a material, component, or supplier problem? (defective raw
    material, vendor/CRO issue, sample integrity)
  - Measurement: Was it a testing, monitoring, or documentation problem?
    (testing error, data integrity, documentation gap)
  - Environment: Was it environmental? (temperature excursion, contamination,
    facility issue, timing/scheduling)

Think through EACH fishbone dimension, then choose the ONE category that is the
PRIMARY root cause. List others as contributing_factors if relevant.

Choose exactly one root_cause_category from:
{categories}

Respond with ONLY a JSON object, no prose, in this exact shape:
{{
  "root_cause_category": "<one category from the list above, verbatim>",
  "root_cause_summary": "<1-3 sentence explanation grounded in the evidence>",
  "contributing_factors": ["<short factor from any fishbone dimension>", "..."],
  "confidence": "high | medium | low"
}}
If the evidence is too thin to determine root cause, use "Root Cause - Not Identified"
with confidence "low"."""

USER_RCA_PROMPT = """<enriched_context>
{context}
</enriched_context>

<raw_deviation_text>
{free_text}
</raw_deviation_text>

Using the evidence above, determine the most likely root cause of this deviation and answer with the JSON object only."""


@dataclass
class RootCauseJudgement:
    """Structured result of one root-cause judgement (one deviation)."""
    root_cause_category: str = "Other / Indeterminate"
    root_cause_summary: str = ""
    contributing_factors: list = field(default_factory=list)
    confidence: str = "low"
    raw_response: str = ""


class RootCauseClassifier:
    """Judges the root cause of a deviation via the Databricks LLM endpoint.

    Encapsulates prompt construction, transient-failure retry, and JSON parsing
    so callers just invoke `.judge(context, free_text)`.
    """

    def __init__(self, client, model, categories, *, max_retries=6,
                 max_tokens=500, temperature=0.1):
        self.client = client
        self.model = model
        self.categories = categories
        self.max_retries = max_retries
        self.max_tokens = max_tokens
        self.temperature = temperature
        self._system_prompt = SYSTEM_RCA_PROMPT.format(
            categories="\n".join(f"- {c}" for c in categories))

    def _parse(self, raw: str) -> RootCauseJudgement:
        """Best-effort parse of the model's JSON reply into a RootCauseJudgement."""
        if not raw:
            return RootCauseJudgement()
        text = re.sub(r"^```(?:json)?|```$", "", raw.strip(), flags=re.IGNORECASE).strip()
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)   # first {...} block
        if not m:
            return RootCauseJudgement(root_cause_summary=raw.strip()[:1000], raw_response=raw)
        try:
            obj = json.loads(m.group(0))
        except json.JSONDecodeError:
            return RootCauseJudgement(root_cause_summary=raw.strip()[:1000], raw_response=raw)
        cats = obj.get("contributing_factors", [])
        if isinstance(cats, str):
            cats = [cats]
        return RootCauseJudgement(
            root_cause_category=str(obj.get("root_cause_category", "Other / Indeterminate")).strip(),
            root_cause_summary=str(obj.get("root_cause_summary", "")).strip(),
            contributing_factors=[str(x).strip() for x in cats if str(x).strip()],
            confidence=str(obj.get("confidence", "low")).strip().lower(),
            raw_response=raw,
        )

    def _call(self, context: str, free_text: str) -> str:
        messages = [
            {"role": "system", "content": self._system_prompt},
            {"role": "user",
             "content": USER_RCA_PROMPT.format(context=context, free_text=free_text)},
        ]
        resp = self.client.chat.completions.create(
            model=self.model, max_tokens=self.max_tokens,
            temperature=self.temperature, messages=messages,
        )
        return (resp.choices[0].message.content or "").strip()

    def judge(self, context: str, free_text: str) -> RootCauseJudgement:
        """Judge one deviation, retrying transient failures with capped backoff."""
        evidence = (context or "").strip() or (free_text or "").strip()
        if not evidence:
            return RootCauseJudgement()
        for attempt in range(self.max_retries + 1):
            try:
                return self._parse(self._call(context, free_text))
            except Exception as e:
                if attempt == self.max_retries:
                    # give up: structured error so one bad row won't kill the run
                    return RootCauseJudgement(raw_response=f"ERROR: {str(e)[:300]}")
                time.sleep(min(2 ** attempt, 60))  # exp backoff, capped at 60s


rca = RootCauseClassifier(client, MODEL_NAME, ROOT_CAUSE_CATEGORIES)

In [0]:
# Cell 4 — Run RCA in parallel and write the root-cause table
# ============================================================================
# ROOT-CAUSE ANALYSIS — run in parallel, assemble, and write the table
# Output: us_gmsgq_dev.gms_us_alyt.deviation_root_cause  (one row per pr_id)
# ============================================================================
results: list = [None] * len(rc_pdf)
lock = threading.Lock()
done = 0

t0 = time.time()
with ThreadPoolExecutor(max_workers=PARALLEL) as ex:
    futs = {
        ex.submit(rca.judge,
                  rc_pdf["contextual_retrieval_text_full"].iat[i],
                  rc_pdf["source_free_text_full"].iat[i]): i
        for i in range(len(rc_pdf))
    }
    for f in as_completed(futs):
        i = futs[f]
        results[i] = f.result()
        with lock:
            done += 1
            if done % 100 == 0:
                print(f"  {done:,}/{len(rc_pdf):,} judged ({time.time()-t0:.0f}s)")

print(f"Judged {len(rc_pdf):,} deviations in {time.time()-t0:.0f}s")

# ---- Assemble (RootCauseJudgement -> columns) -------------------------------
rc_out = pd.DataFrame({
    "pr_id":                 rc_pdf["pr_id"].values,
    "root_cause_category":   [j.root_cause_category  for j in results],
    "root_cause_summary":    [j.root_cause_summary   for j in results],
    "contributing_factors":  [j.contributing_factors for j in results],
    "confidence":            [j.confidence           for j in results],
    "raw_response":          [j.raw_response         for j in results],
})

print("\nRoot-cause category distribution:")
print(rc_out["root_cause_category"].value_counts().to_string())

# ---- Write (one row per pr_id) ---------------------------------------------
rc_schema = T.StructType([
    T.StructField("pr_id",                T.StringType(),              False),
    T.StructField("root_cause_category",  T.StringType(),              True),
    T.StructField("root_cause_summary",   T.StringType(),              True),
    T.StructField("contributing_factors", T.ArrayType(T.StringType()), True),
    T.StructField("confidence",           T.StringType(),              True),
    T.StructField("raw_response",         T.StringType(),              True),
])
rc_rows = [
    (r.pr_id, r.root_cause_category, r.root_cause_summary,
     list(r.contributing_factors), r.confidence, r.raw_response)
    for r in rc_out.itertuples(index=False)
]
rc_sdf = spark.createDataFrame(rc_rows, schema=rc_schema)
(rc_sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(RC_TABLE))

print(f"\n{RC_TABLE}: {rc_sdf.count():,} rows (one root-cause judgement per pr_id)")
display(rc_sdf.limit(10))

In [0]:
# Cell 5 — Unsupervised BERTopic root-cause clusters
# ============================================================================
# BERTopic UNSUPERVISED — Discover natural root-cause clusters from embeddings
#
# Clusters the precomputed BGE-M3 embeddings (contextual_retrieval_text_full)
# via UMAP + HDBSCAN. c-TF-IDF labels clusters with keywords from the clean
# source_free_text_full. Includes multi-label via approximate_distribution().
# ============================================================================

# ---- Config (best params from topic_modeling grid search) ----
BEST_NN, BEST_NC, BEST_MD = 5, 5, 0.0

STRUCTURAL_STOPS = [
    "event", "title", "description", "impact", "assessment", "quality", "final",
    "root", "cause", "category", "subcategory", "action", "text",
    "resolved", "references", "canonical", "vendor", "external", "supplier",
    "country", "acronym", "definitions", "cro", "system", "name", "document",
    "device", "enrichment", "clinical", "id", "type", "number",
]
custom_stops = list(ENGLISH_STOP_WORDS) + STRUCTURAL_STOPS

# ---- Fit BERTopic ----
rc_topic_model = BERTopic(
    embedding_model=None,
    umap_model=UMAP(n_neighbors=BEST_NN, n_components=BEST_NC, min_dist=BEST_MD,
                    metric="cosine", random_state=42),
    hdbscan_model=HDBSCAN(min_cluster_size=10, min_samples=5,
                          metric="euclidean", cluster_selection_method="eom",
                          prediction_data=True),
    vectorizer_model=CountVectorizer(stop_words=custom_stops, min_df=2, ngram_range=(1, 2)),
    calculate_probabilities=True,
    verbose=True,
)

rc_topics, _ = rc_topic_model.fit_transform(rc_docs, embeddings=rc_embs)

rc_info = rc_topic_model.get_topic_info()
n_topics = len(rc_info[rc_info.Topic != -1])
n_outlier = int((np.array(rc_topics) == -1).sum())
print(f"Topics: {n_topics}  |  Outliers: {n_outlier} ({100*n_outlier/len(rc_topics):.1f}%)")

# ---- Topic summary with keywords ----
topic_short_labels = {}
for _, row in rc_info.iterrows():
    t = row["Topic"]
    if t == -1:
        topic_short_labels[t] = "Outlier"
    else:
        kws = [w for w, _ in rc_topic_model.get_topic(t)[:5]]
        topic_short_labels[t] = " / ".join(kws[:3])

display(rc_info.head(30))

# ---- Multi-label assignment ----
print("\nComputing multi-label assignment (approximate_distribution)...")
rc_distr, _ = rc_topic_model.approximate_distribution(rc_docs, min_similarity=0.01)

MULTI_THRESHOLD = 0.1
rc_multi = []
for i in range(len(rc_docs)):
    doc_probs = rc_distr[i]
    above = [(t, float(doc_probs[t])) for t in range(len(doc_probs)) if doc_probs[t] >= MULTI_THRESHOLD]
    above.sort(key=lambda x: -x[1])
    rc_multi.append({
        "pr_id": emb_pdf["pr_id"].values[i],
        "n_topics": len(above),
        "primary_topic": rc_topics[i],
        "primary_label": topic_short_labels.get(rc_topics[i], "Outlier"),
        "all_topics": [t for t, _ in above],
        "all_labels": [topic_short_labels.get(t, f"Topic {t}") for t, _ in above],
        "all_probs": [p for _, p in above],
    })

rc_multi_df = pd.DataFrame(rc_multi)
print(f"Events with 1 topic: {(rc_multi_df['n_topics'] == 1).sum()}")
print(f"Events with 2+ topics: {(rc_multi_df['n_topics'] >= 2).sum()}")
print(f"Average topics/event: {rc_multi_df['n_topics'].mean():.2f}")

# ---- 3D UMAP scatter ----
umap_3d = UMAP(n_neighbors=BEST_NN, n_components=3, min_dist=BEST_MD,
               metric="cosine", random_state=42)
coords = umap_3d.fit_transform(rc_embs)

# pr_id aligned with rc_docs/rc_embs (all rows via emb_pdf)
plot_df = pd.DataFrame({
    "UMAP-1": coords[:, 0], "UMAP-2": coords[:, 1], "UMAP-3": coords[:, 2],
    "topic": [topic_short_labels.get(t, f"Topic {t}") for t in rc_topics],
    "pr_id": emb_pdf["pr_id"].values,
    "ground_truth_rc": emb_pdf["pr_id"].map(gt_lookup).values,
    "event": [d[:200] for d in rc_docs],
})

fig = px.scatter_3d(
    plot_df, x="UMAP-1", y="UMAP-2", z="UMAP-3", color="topic",
    hover_data={"pr_id": True, "ground_truth_rc": True, "event": True,
                "UMAP-1": False, "UMAP-2": False, "UMAP-3": False},
    title=f"BERTopic Unsupervised Root-Cause Clusters — {n_topics} topics",
    opacity=0.75, height=700,
)
fig.update_traces(marker_size=3.5)
fig.show()

In [0]:
# Cell 6 — Guided BERTopic (Ishikawa-seeded)
# ============================================================================
# GUIDED BERTopic — Seeded from Ishikawa root-cause categories
#
# Uses the ground-truth Root_Cause_Category + SubCategory values as seed topics.
# BERTopic nudges clusters toward these seeds while discovering emergent ones.
# Includes multi-label via approximate_distribution().
# ============================================================================

# ---- Build seed keyword lists from ground-truth categories ----
ISHIKAWA_SEEDS = {
    "Method": ["method", "procedure", "sop", "process", "step", "protocol", "instruction", "sampling"],
    "Human Factors": ["human", "error", "mistake", "distraction", "fatigue", "oversight", "violation", "personnel"],
    "Machine": ["machine", "equipment", "instrument", "hardware", "software", "malfunction", "calibration", "system"],
    "Environment": ["environment", "temperature", "humidity", "contamination", "conditions", "facility", "power", "utility"],
    "Material": ["material", "sample", "reagent", "component", "packaging", "storage", "shipping", "degradation"],
    "Measurement": ["measurement", "test", "assay", "specification", "oos", "result", "analytical", "laboratory"],
}

# Enrich seeds with SubCategory keywords from ground truth
for _, row in gt_pdf.iterrows():
    cat = row["Root_Cause_Category"].strip()
    sub = row["Root_Cause_SubCategory"].strip()
    if cat in ISHIKAWA_SEEDS and sub:
        for w in sub.lower().replace("/", " ").replace("-", " ").split():
            if len(w) >= 4 and w not in {"with", "that", "from", "were", "this", "been", "have"}:
                ISHIKAWA_SEEDS[cat].append(w)

# De-duplicate and cap
seed_topic_list = []
seed_labels = []
for cat, words in ISHIKAWA_SEEDS.items():
    unique_words = list(dict.fromkeys(words))[:15]  # preserve order, cap at 15
    seed_topic_list.append(unique_words)
    seed_labels.append(cat)

print(f"Seed topics ({len(seed_labels)}):")
for label, seeds in zip(seed_labels, seed_topic_list):
    print(f"  {label:<20} seeds: {seeds[:10]}")

# ---- Fit Guided BERTopic ----
guided_rc_model = BERTopic(
    embedding_model=None,
    umap_model=UMAP(n_neighbors=BEST_NN, n_components=BEST_NC, min_dist=BEST_MD,
                    metric="cosine", random_state=42),
    hdbscan_model=HDBSCAN(min_cluster_size=10, min_samples=5,
                          metric="euclidean", cluster_selection_method="eom",
                          prediction_data=True),
    vectorizer_model=CountVectorizer(stop_words=custom_stops, min_df=2, ngram_range=(1, 2)),
    seed_topic_list=seed_topic_list,
    calculate_probabilities=True,
    verbose=True,
)

guided_rc_topics, _ = guided_rc_model.fit_transform(rc_docs, embeddings=rc_embs)

guided_info = guided_rc_model.get_topic_info()
n_guided = len(guided_info[guided_info.Topic != -1])
n_out = int((np.array(guided_rc_topics) == -1).sum())
print(f"\nGuided topics: {n_guided}  |  Outliers: {n_out} ({100*n_out/len(guided_rc_topics):.1f}%)")
print(f"  Seeded: {min(n_guided, len(seed_labels))}  |  Emergent: {max(0, n_guided - len(seed_labels))}")

# ---- Summary: Topic → Seed Category → Keywords ----
guided_summary = []
for _, row in guided_info.iterrows():
    t = row["Topic"]
    if t == -1:
        guided_summary.append({"Topic": -1, "Seed Category": "(Outlier)",
                               "Count": row["Count"], "Keywords": ""})
    else:
        kws = [w for w, _ in guided_rc_model.get_topic(t)[:8]]
        seed_cat = seed_labels[t] if t < len(seed_labels) else "✨ Emergent"
        guided_summary.append({
            "Topic": t, "Seed Category": seed_cat,
            "Count": row["Count"], "Keywords": ", ".join(kws),
        })

guided_rc_df = pd.DataFrame(guided_summary)
print("\n" + "="*90)
print("GUIDED ROOT-CAUSE TOPICS  (seeded from Ishikawa categories)")
print("="*90)
display(guided_rc_df)

# ---- Multi-label ----
print("\nComputing multi-label (guided model)...")
guided_distr, _ = guided_rc_model.approximate_distribution(rc_docs, min_similarity=0.01)

guided_multi = []
for i in range(len(rc_docs)):
    doc_probs = guided_distr[i]
    above = [(t, float(doc_probs[t])) for t in range(len(doc_probs)) if doc_probs[t] >= MULTI_THRESHOLD]
    above.sort(key=lambda x: -x[1])
    guided_multi.append({
        "pr_id": emb_pdf["pr_id"].values[i],
        "n_root_causes": len(above),
        "primary_rc": seed_labels[guided_rc_topics[i]] if 0 <= guided_rc_topics[i] < len(seed_labels) else "Emergent" if guided_rc_topics[i] >= 0 else "Outlier",
        "all_rc_labels": [seed_labels[t] if t < len(seed_labels) else f"Emergent-{t}" for t, _ in above],
        "all_rc_probs": [p for _, p in above],
    })

guided_multi_df = pd.DataFrame(guided_multi)
print(f"Events with 1 root cause: {(guided_multi_df['n_root_causes'] == 1).sum()}")
print(f"Events with 2+ root causes: {(guided_multi_df['n_root_causes'] >= 2).sum()}")
print(f"Average root causes/event: {guided_multi_df['n_root_causes'].mean():.2f}")

# ---- 3D scatter coloured by guided root cause ----
guided_labels_map = {}
for _, row in guided_rc_df.iterrows():
    t = int(row["Topic"])
    guided_labels_map[t] = "Outlier" if t == -1 else row["Seed Category"]

# pr_id aligned with rc_docs/rc_embs (all rows via emb_pdf)
plot_df2 = pd.DataFrame({
    "UMAP-1": coords[:, 0], "UMAP-2": coords[:, 1], "UMAP-3": coords[:, 2],
    "root_cause": [guided_labels_map.get(t, f"Topic {t}") for t in guided_rc_topics],
    "pr_id": emb_pdf["pr_id"].values,
    "ground_truth_rc": emb_pdf["pr_id"].map(gt_lookup).values,
    "event": [d[:200] for d in rc_docs],
})

fig2 = px.scatter_3d(
    plot_df2, x="UMAP-1", y="UMAP-2", z="UMAP-3", color="root_cause",
    hover_data={"pr_id": True, "ground_truth_rc": True, "event": True,
                "UMAP-1": False, "UMAP-2": False, "UMAP-3": False},
    title=f"Guided BERTopic Root Cause — Ishikawa Seeded ({n_guided} topics)",
    opacity=0.75, height=700,
)
fig2.update_traces(marker_size=3.5)
fig2.show()

In [0]:
# Cell 7 — Zero-shot BERTopic (SubCategory labels)
# ============================================================================
# ZERO-SHOT BERTopic — Root-Cause SubCategory as candidate labels
#
# Uses the most common Root_Cause_SubCategory values as candidate topic labels.
# Assigns docs by cosine similarity in BGE-M3 space; unmatched go to clustering.
# Uses the BGE-M3 wrapper (BaseEmbedder) to encode labels in the same space.
# Includes multi-label via approximate_distribution().
# ============================================================================

class BGEM3Wrapper(BaseEmbedder):
    """BERTopic-compatible BGE-M3 wrapper for encoding candidate labels."""
    def __init__(self, model_name="BAAI/bge-m3", use_fp16=True):
        super().__init__()
        self.model = BGEM3FlagModel(model_name, use_fp16=use_fp16)
        self.embedding_dimension = 1024

    def embed(self, documents, verbose=False):
        output = self.model.encode(
            documents, batch_size=32, max_length=512,
            return_dense=True, return_sparse=False, return_colbert_vecs=False,
        )
        return np.array(output["dense_vecs"], dtype=np.float32)

# ---- Build candidate labels from most common SubCategories ----
sub_counts = (
    gt_pdf[gt_pdf["Root_Cause_SubCategory"].str.strip() != ""]
    .groupby("Root_Cause_SubCategory").size()
    .sort_values(ascending=False)
)

# Take top subcategories that cover meaningful population
rc_candidate_labels = sub_counts.head(20).index.tolist()
print(f"Zero-shot candidate labels ({len(rc_candidate_labels)}):")
for i, lbl in enumerate(rc_candidate_labels):
    print(f"  {i+1:>2}. {lbl}")

# ---- Load BGE-M3 for label encoding ----
print("\nLoading BGE-M3 for label encoding...")
bge_wrapper = BGEM3Wrapper("BAAI/bge-m3", use_fp16=True)
print(f"BGE-M3 loaded  |  dim={bge_wrapper.embedding_dimension}")

# ---- Fit Zero-Shot BERTopic ----
# Note: with 20 zero-shot labels at threshold=0.45, most docs match a label,
# leaving very few residual docs for clustering. Use minimal params so
# HDBSCAN/UMAP can handle even 1-2 residual docs without error.
zs_rc_model = BERTopic(
    embedding_model=bge_wrapper,
    umap_model=UMAP(n_neighbors=2, n_components=min(BEST_NC, 2), min_dist=BEST_MD,
                    metric="cosine", random_state=42),
    hdbscan_model=HDBSCAN(min_cluster_size=2, min_samples=1,
                          metric="euclidean", cluster_selection_method="eom",
                          prediction_data=True),
    vectorizer_model=CountVectorizer(stop_words=custom_stops, min_df=2, ngram_range=(1, 2)),
    zeroshot_topic_list=rc_candidate_labels,
    zeroshot_min_similarity=0.45,
    calculate_probabilities=True,
    verbose=True,
)

zs_rc_topics, _ = zs_rc_model.fit_transform(rc_docs, embeddings=rc_embs)

zs_rc_info = zs_rc_model.get_topic_info()
n_zs = len(zs_rc_info[zs_rc_info.Topic != -1])
n_out = int((np.array(zs_rc_topics) == -1).sum())
print(f"\nZero-shot results:  Topics: {n_zs}  |  Outliers: {n_out} ({100*n_out/len(zs_rc_topics):.1f}%)")

# ---- Summary ----
zs_rc_summary = []
for _, row in zs_rc_info.iterrows():
    t = row["Topic"]
    if t == -1:
        zs_rc_summary.append({"Topic": -1, "Label": "(Outlier)", "Count": row["Count"],
                              "Keywords": "", "Source": ""})
    else:
        kws = [w for w, _ in zs_rc_model.get_topic(t)[:8]]
        label = row.get("Name", "") or f"Topic {t}"
        source = "Zero-shot" if t < len(rc_candidate_labels) else "Emergent"
        zs_rc_summary.append({
            "Topic": t, "Label": label, "Count": row["Count"],
            "Keywords": ", ".join(kws), "Source": source,
        })

zs_rc_df = pd.DataFrame(zs_rc_summary)
print("\n" + "="*95)
print("ZERO-SHOT ROOT-CAUSE TOPICS  (BGE-M3 embeddings, SubCategory labels)")
print("="*95)
display(zs_rc_df)

# ---- Multi-label ----
print("\nComputing multi-label (zero-shot model)...")
zs_rc_distr, _ = zs_rc_model.approximate_distribution(rc_docs, min_similarity=0.01)

zs_rc_multi = []
zs_labels_map = {}
for _, row in zs_rc_df.iterrows():
    t = int(row["Topic"])
    zs_labels_map[t] = "Outlier" if t == -1 else row["Label"]

for i in range(len(rc_docs)):
    doc_probs = zs_rc_distr[i]
    above = [(t, float(doc_probs[t])) for t in range(len(doc_probs)) if doc_probs[t] >= MULTI_THRESHOLD]
    above.sort(key=lambda x: -x[1])
    zs_rc_multi.append({
        "pr_id": emb_pdf["pr_id"].values[i],
        "n_root_causes": len(above),
        "primary_rc": zs_labels_map.get(zs_rc_topics[i], "Outlier"),
        "all_rc_labels": [zs_labels_map.get(t, f"Topic {t}") for t, _ in above],
        "all_rc_probs": [p for _, p in above],
    })

zs_rc_multi_df = pd.DataFrame(zs_rc_multi)
print(f"Events with 1 root cause: {(zs_rc_multi_df['n_root_causes'] == 1).sum()}")
print(f"Events with 2+ root causes: {(zs_rc_multi_df['n_root_causes'] >= 2).sum()}")
print(f"Average root causes/event: {zs_rc_multi_df['n_root_causes'].mean():.2f}")

# ---- 3D scatter ----
# pr_id aligned with rc_docs/rc_embs (all rows via emb_pdf)
plot_df3 = pd.DataFrame({
    "UMAP-1": coords[:, 0], "UMAP-2": coords[:, 1], "UMAP-3": coords[:, 2],
    "root_cause": [zs_labels_map.get(t, f"Topic {t}") for t in zs_rc_topics],
    "pr_id": emb_pdf["pr_id"].values,
    "ground_truth_rc": emb_pdf["pr_id"].map(gt_lookup).values,
    "event": [d[:200] for d in rc_docs],
})

fig3 = px.scatter_3d(
    plot_df3, x="UMAP-1", y="UMAP-2", z="UMAP-3", color="root_cause",
    hover_data={"pr_id": True, "ground_truth_rc": True, "event": True,
                "UMAP-1": False, "UMAP-2": False, "UMAP-3": False},
    title=f"Zero-Shot BERTopic Root Cause — SubCategory Labels ({n_zs} topics)",
    opacity=0.75, height=700,
)
fig3.update_traces(marker_size=3.5)
fig3.show()

In [0]:
# Cell 8 — Compare LLM + zero-shot vs ground truth
# ============================================================================
# COMPARISON — LLM + Zero-Shot root-cause vs ground-truth
#
# Models kept:
#   1. LLM (Ishikawa 6M prompt): primary from root_cause_category,
#      secondary/additional from contributing_factors
#   2. Zero-Shot BERTopic (BGE-M3): primary/secondary/additional from
#      approximate_distribution() ranked probabilities
# ============================================================================

# ---- Ground truth (deduplicated to 1 row per pr_id) ----
_gt_dedup = gt_lookup.reset_index().rename(columns={"Root_Cause_Category": "ground_truth"})

# ---- LLM: primary + secondary + additional from contributing_factors ----
try:
    llm_comp = rc_out[["pr_id", "root_cause_category", "contributing_factors"]].copy()
    llm_comp = llm_comp.rename(columns={"root_cause_category": "LLM_primary"})
    llm_comp["LLM_secondary"] = llm_comp["contributing_factors"].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) > 0 else ""
    )
    llm_comp["LLM_tertiary"] = llm_comp["contributing_factors"].apply(
        lambda x: x[1] if isinstance(x, list) and len(x) > 1 else ""
    )
    llm_comp = llm_comp[["pr_id", "LLM_primary", "LLM_secondary", "LLM_tertiary"]]
    print(f"LLM results: {len(llm_comp):,} events")
except NameError:
    llm_comp = pd.DataFrame({"pr_id": rc_pdf["pr_id"].values,
                             "LLM_primary": None, "LLM_secondary": "", "LLM_tertiary": ""})
    print("(LLM results not in memory — run cells 3-4 first)")

# ---- Zero-Shot BERTopic: primary + secondary + additional from all_rc_labels ----
zs_comp = zs_rc_multi_df[["pr_id", "all_rc_labels"]].copy()
zs_comp["ZS_primary"] = zs_comp["all_rc_labels"].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else ""
)
zs_comp["ZS_secondary"] = zs_comp["all_rc_labels"].apply(
    lambda x: x[1] if isinstance(x, list) and len(x) > 1 else ""
)
zs_comp["ZS_tertiary"] = zs_comp["all_rc_labels"].apply(
    lambda x: x[2] if isinstance(x, list) and len(x) > 2 else ""
)
zs_comp = zs_comp[["pr_id", "ZS_primary", "ZS_secondary", "ZS_tertiary"]]

# ---- Add event text ----
text_df = rc_pdf[["pr_id", "source_free_text_full"]].copy()
text_df = text_df.rename(columns={"source_free_text_full": "text"})

# ---- Assemble comparison ----
comp = _gt_dedup.merge(llm_comp, on="pr_id", how="right")
comp = comp.merge(zs_comp, on="pr_id", how="left")
comp = comp.merge(text_df, on="pr_id", how="left")

print(f"\nTotal events: {len(comp):,}")
print("\n" + "="*80)
print("SAMPLE: PRIMARY + SECONDARY + TERTIARY ROOT CAUSES")
print("="*80)

sample = comp.sample(n=20, random_state=42)
display(
    sample[[
        "pr_id", "text", "ground_truth",
        "ZS_primary", "ZS_secondary", "ZS_tertiary",
        "LLM_primary", "LLM_secondary", "LLM_tertiary",
    ]]
)

# ---- Agreement metrics (where ground truth exists) ----
valid = comp[comp["ground_truth"].notna() & (comp["ground_truth"].str.strip() != "")].copy()
print(f"\nEvents with ground-truth: {len(valid):,} / {len(comp):,}")
print("\n" + "="*80)
print("MODEL AGREEMENT WITH GROUND TRUTH (primary root cause)")
print("="*80)

# LLM exact match (already Ishikawa-aligned after prompt update)
if valid["LLM_primary"].notna().any():
    # Map LLM categories to match ground-truth naming
    LLM_TO_GT = {
        "Man (Human Factors)": "Human Factors",
        "Method": "Method",
        "Machine": "Machine",
        "Material": "Material",
        "Measurement": "Measurement",
        "Environment": "Environment",
        "Root Cause - Not Identified": "Root Cause - Not Identified",
        # Old taxonomy mappings (in case old results are still in memory)
        "Human Error": "Human Factors",
        "Procedure / SOP Gap": "Method",
        "Training / Competency": "Human Factors",
        "Equipment / System Failure": "Machine",
        "Documentation Error": "Measurement",
        "Communication / Handoff": "Human Factors",
        "Vendor / Supplier / CRO": "Material",
        "Material / Sample Issue": "Material",
        "Scheduling / Timing": "Environment",
        "Process Design": "Method",
        "Other / Indeterminate": "Root Cause - Not Identified",
    }
    valid["llm_mapped"] = valid["LLM_primary"].map(LLM_TO_GT).fillna(valid["LLM_primary"])
    llm_match = (valid["llm_mapped"] == valid["ground_truth"]).mean()
    print(f"  LLM exact match:        {llm_match:.1%}")

# Zero-Shot: primary_rc is already a SubCategory label, not directly comparable
# to the 6 Ishikawa categories. Compare at cluster level via ARI/NMI.
print("\n  (Zero-Shot uses SubCategory labels — not directly comparable to 6M categories)")
print("  Using ARI/NMI for clustering agreement:")

gt_encoded = LabelEncoder().fit_transform(valid["ground_truth"])

for col, label in [("llm_mapped", "LLM (Ishikawa)"), ("ZS_primary", "Zero-Shot (SubCat)")]:
    if col in valid.columns and valid[col].notna().any():
        pred_encoded = LabelEncoder().fit_transform(valid[col].fillna("Unknown"))
        ari = adjusted_rand_score(gt_encoded, pred_encoded)
        nmi = normalized_mutual_info_score(gt_encoded, pred_encoded)
        print(f"    {label:<25} ARI={ari:.4f}  NMI={nmi:.4f}")

# ---- Distribution ----
print("\n" + "="*80)
print("PRIMARY ROOT-CAUSE DISTRIBUTION")
print("="*80)
print("\nGround truth:")
print(valid["ground_truth"].value_counts().to_string())
print("\nLLM:")
print(comp["LLM_primary"].value_counts().head(15).to_string())
print("\nZero-Shot:")
print(comp["ZS_primary"].value_counts().head(15).to_string())